<a href="https://colab.research.google.com/github/relugzosiraba/Juqbox.jl/blob/master/campaign.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nonlocal LZ Predictor Campaign
**Runtime → Change runtime type → TPU**

Tests the nonlocal Landau-Zener predictor across system sizes L=5-12.

In [1]:
!pip install -q jax[tpu] -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
!pip install -q scipy

import jax
print(f'JAX backend: {jax.default_backend()}')
print(f'JAX devices: {jax.devices()}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.1/155.1 MB 4.4 MB/s eta 0:00:00
JAX backend: gpu
JAX devices: [CudaDevice(id=0)]


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from scipy.stats import spearmanr, pearsonr
from scipy.linalg import expm, eigh
import time
import json

H0 = 10.0

sx = np.array([[0,1],[1,0]], dtype=complex)
sy = np.array([[0,-1j],[1j,0]], dtype=complex)
sz = np.array([[1,0],[0,-1]], dtype=complex)
si = np.eye(2, dtype=complex)

def kron_chain(ops):
    r = np.array([[1.0]], dtype=complex)
    for o in ops:
        r = np.kron(r, o)
    return r

def build_sigma(op, site, L):
    ops = [si]*L
    ops[site] = op
    return kron_chain(ops)

def build_H_initial(L):
    return sum(H0 * build_sigma(sx, k, L) for k in range(L))

def build_H_final(L, h_fields, J):
    dim = 2**L
    H = np.zeros((dim, dim), dtype=complex)
    for k in range(L):
        H += h_fields[k] * build_sigma(sz, k, L)
    for k in range(L):
        kp1 = (k+1) % L
        ops = [si]*L; ops[k] = sz; ops[kp1] = sz
        H += J * kron_chain(ops)
    return H

def compute_steering(h_fields, J, L, s):
    fi = np.cos(np.pi*s/2)**2; ff = np.sin(np.pi*s/2)**2
    dfi = -np.pi*np.cos(np.pi*s/2)*np.sin(np.pi*s/2)
    dff = np.pi*np.cos(np.pi*s/2)*np.sin(np.pi*s/2)
    dim = 2**L
    H_steer = np.zeros((dim, dim), dtype=complex)
    for k in range(L):
        Bx = H0*fi; Bz = h_fields[k]*ff
        dBx = H0*dfi; dBz = h_fields[k]*dff
        B = np.array([Bx, 0, Bz]); dB = np.array([dBx, 0, dBz])
        B2 = np.dot(B, B)
        if B2 < 1e-20: continue
        cross = np.cross(B, dB)
        coeff = cross / (2.0 * B2)
        H_steer += coeff[0]*build_sigma(sx,k,L) + coeff[1]*build_sigma(sy,k,L) + coeff[2]*build_sigma(sz,k,L)
    return H_steer

def run_annealing(L, h_fields, J, t_a, n_steps=100, use_steering=False):
    H_i = build_H_initial(L); H_f = build_H_final(L, h_fields, J)
    E_i, V_i = eigh(H_i); psi = V_i[:,0].astype(complex)
    E_f, V_f = eigh(H_f); gs_final = V_f[:,0]
    dt = t_a / n_steps
    for step in range(n_steps):
        s = (step+0.5)/n_steps
        fi = np.cos(np.pi*s/2)**2; ff = np.sin(np.pi*s/2)**2
        H = fi*H_i + ff*H_f
        if use_steering:
            H += compute_steering(h_fields, J, L, s) / t_a
        psi = expm(-1j*H*dt) @ psi
    return np.abs(np.vdot(gs_final, psi))**2

def compute_nonlocal_lz(h_fields, J, L):
    dim = 2**L
    H_i = build_H_initial(L)
    H_f_local = sum(h_fields[k]*build_sigma(sz,k,L) for k in range(L))
    zz_list = []
    for k in range(L):
        ops = [si]*L; ops[k] = sz; ops[(k+1)%L] = sz
        zz_list.append(kron_chain(ops))
    H_f_int = J * sum(zz_list)
    H_f = H_f_local + H_f_int
    n_s = 50; s_vals = np.linspace(0.05, 0.95, n_s); ds = s_vals[1]-s_vals[0]; eps = 0.05
    total = 0.0
    for s in s_vals:
        fi = np.cos(np.pi*s/2)**2; ff = np.sin(np.pi*s/2)**2
        dfi = -np.pi*np.cos(np.pi*s/2)*np.sin(np.pi*s/2)
        dff = np.pi*np.cos(np.pi*s/2)*np.sin(np.pi*s/2)
        H = fi*H_i + ff*H_f
        evals, evecs = eigh(H)
        gs = evecs[:,0]; ex1 = evecs[:,1]; gap = evals[1]-evals[0]
        dHds_local = dfi*H_i + dff*H_f_local
        dHds_full = dfi*H_i + dff*H_f
        me_local = np.abs(np.vdot(ex1, dHds_local@gs))**2
        me_full = np.abs(np.vdot(ex1, dHds_full@gs))**2
        nf = np.clip(1.0 - me_local/max(me_full, 1e-20), 0, 1)
        total += nf * me_full / (gap**2 + eps) * ds
    return -np.log(max(total, 1e-12))

print('Physics code loaded ✓')

In [ ]:
J = 0.30
t_a = 10.0
seed = 2026

configs = [
    (5,  100),
    (6,  50),
    (8,  20),
    (10, 10),
    (12, 5),
]

all_results = {}

for L, N in configs:
    print(f'\n{"="*60}')
    print(f'L={L}, N={N}, dim={2**L}')
    print(f'{"="*60}', flush=True)

    np.random.seed(seed)
    preds = []; dPs = []
    t0 = time.time()

    for trial in range(N):
        elapsed = time.time() - t0
        h = np.random.uniform(-1, 1, L)
        pred = compute_nonlocal_lz(h, J, L)
        preds.append(pred)
        Pb = run_annealing(L, h, J, t_a, n_steps=100, use_steering=False)
        Ps = run_annealing(L, h, J, t_a, n_steps=100, use_steering=True)
        dP = Ps - Pb
        dPs.append(dP)
        print(f'  {trial+1}/{N} ({elapsed:.0f}s) dP={dP:+.4f} pred={pred:.4f}', flush=True)
        if L >= 10 and elapsed > 7200:
            print('  TIME LIMIT'); break

    if len(dPs) >= 3:
        rho, _ = spearmanr(preds, dPs)
        r, _ = pearsonr(preds, dPs)
    else:
        rho, r = 0.0, 0.0

    result = {'L': L, 'J': J, 'N': len(dPs), 'rho': float(rho), 'r': float(r),
              'mean_dP': float(np.mean(dPs)), 'frac_pos': float(np.mean(np.array(dPs)>0)),
              'time': time.time()-t0}
    all_results[L] = result
    print(f'\n  *** L={L}: rho={rho:.4f}, r={r:.4f}, <dP>={np.mean(dPs):+.4f} ***')

print(f'\n{"="*60}')
print('FINAL SUMMARY')
print(f'{"="*60}')
print(f'{"L":>4} {"N":>4} {"rho":>8} {"r":>8} {"<dP>":>8} {"time":>8}')
print('-'*45)
for L in sorted(all_results.keys()):
    r = all_results[L]
    print(f'{L:>4} {r["N"]:>4} {r["rho"]:>8.4f} {r["r"]:>8.4f} {r["mean_dP"]:>8.4f} {r["time"]:>8.1f}s')